# 13 · RDKit native metal stereochemistry

- Works: RDKit parses and writes @SPn, @TBn, and @OHn, and can perceive them from suitable 3D
  coordinates.
- Partially works: a native tag can bias some embeddings toward a useful structure.
- Fails silently: ETKDG does not reliably enforce non-tetrahedral tags; the stored tag can disagree
  with the coordinates.
- Missing natively: symmetry-reduced coordination-isomer enumeration, haptic identity, and lossless
  multimetal arrangement round-trips.
- rxembed keeps the object as an RDKit Mol and supplies those missing coordination layers.

Sources: [RDKit Book](https://rdkit.org/docs/RDKit_Book.html#support-for-non-tetrahedral-atomic-stereochemistry) and [RDKit issue 7948](https://github.com/rdkit/rdkit/issues/7948).

In [ ]:
from rdkit import Chem, rdBase
from rdkit.Chem import rdDistGeom, rdMolAlign, rdMolTransforms

import rxembed as rx

Chem.SetAllowNontetrahedralChirality(True)
print("RDKit", rdBase.rdkitVersion, "rxembed", rx.__version__)

## Native tags can help one embedding

- The graph and random seed are identical.
- Plain ETKDG folds one donor substituent toward Ni.
- @SP3 avoids that fold here. This is useful, but it is not general non-tetrahedral enforcement.

In [ ]:
def native_embed(smiles, seed):
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    params = rdDistGeom.ETKDGv3()
    params.randomSeed = seed
    with rdBase.BlockLogs():
        code = rdDistGeom.EmbedMolecule(mol, params)
    return code, mol


def min_mdx(mol):
    metal = next(atom.GetIdx() for atom in mol.GetAtoms() if atom.GetSymbol() == "Ni")
    conf = mol.GetConformer()
    return min(
        rdMolTransforms.GetAngleDeg(conf, metal, donor.GetIdx(), neighbour.GetIdx())
        for donor in mol.GetAtomWithIdx(metal).GetNeighbors()
        for neighbour in donor.GetNeighbors()
        if neighbour.GetIdx() != metal
    )


ni_sp3 = "C[P]1(C)CC[P](C)(C)->[Ni@SP3+2]<-12<-[O-]C(=O)C[N-]->2C"
native_ni = {}
for label, smiles in (("plain", ni_sp3.replace("@SP3", "")), ("@SP3", ni_sp3)):
    code, native_ni[label] = native_embed(smiles, seed=19)
    print(f"{label:5s}: return {code}, minimum Ni-D-X angle {min_mdx(native_ni[label]):.1f} degrees")

## A stored tag is not proof of the generated arrangement

- OH1 and OH2 are opposite arrangements for this fixed neighbour order.
- Both embeddings return success and retain their requested atom tag.
- With the same seed they produce identical coordinates.
- Clear the stored tag and perceive from 3D: both structures are OH1. The OH2 result is a silent
  failure.

In [ ]:
def native_tag(mol, center):
    atom = mol.GetAtomWithIdx(center)
    kind = {
        Chem.ChiralType.CHI_SQUAREPLANAR: "SP",
        Chem.ChiralType.CHI_TRIGONALBIPYRAMIDAL: "TB",
        Chem.ChiralType.CHI_OCTAHEDRAL: "OH",
    }.get(atom.GetChiralTag())
    number = atom.GetUnsignedProp("_chiralPermutation") if atom.HasProp("_chiralPermutation") else 0
    return f"{kind}{number}" if kind else "none"


def perceive_from_3d(mol, center):
    perceived = Chem.Mol(mol)
    atom = perceived.GetAtomWithIdx(center)
    atom.SetChiralTag(Chem.ChiralType.CHI_UNSPECIFIED)
    if atom.HasProp("_chiralPermutation"):
        atom.ClearProp("_chiralPermutation")
    Chem.AssignStereochemistryFrom3D(perceived, confId=0, replaceExistingTags=True)
    return native_tag(perceived, center)


native_oh = {}
for number in (1, 2):
    code, mol = native_embed(f"O[Co@OH{number}](Cl)(C)(N)(F)P", seed=7)
    center = next(atom.GetIdx() for atom in mol.GetAtoms() if atom.GetChiralTag() == Chem.ChiralType.CHI_OCTAHEDRAL)
    native_oh[number] = mol
    print(f"OH{number}: return {code}, stored {native_tag(mol, center)}, perceived {perceive_from_3d(mol, center)}")

rmsd = rdMolAlign.AlignMol(Chem.Mol(native_oh[2]), native_oh[1])
print(f"OH1/OH2 coordinate RMSD: {rmsd:.6f} A")

## Native permutation tags are not chemical enumeration

- RDKit writes all 30 OHn values for six distinct ligands.
- It also writes 30 values for six identical ammines, although these are one chemical arrangement.
- rxembed reduces assignments by ligand graph symmetry and proper polyhedron rotations.

In [ ]:
def native_oh_strings(template):
    return {Chem.CanonSmiles(template.format(number)) for number in range(1, 31)}


native_distinct = native_oh_strings("O->[Co@OH{}+3](<-[Cl-])(<-[CH3-])(<-N)(<-[F-])<-P")
native_identical = native_oh_strings("[NH3]->[Co@OH{}+3](<-[NH3])(<-[NH3])(<-[NH3])(<-[NH3])<-[NH3]")

rx_distinct = rx.metal("O->[Co+3](<-[Cl-])(<-[CH3-])(<-N)(<-[F-])<-P", "OCT", stereo="free")
rx_identical = rx.metal("[NH3]->[Co+3](<-[NH3])(<-[NH3])(<-[NH3])(<-[NH3])<-[NH3]", "OCT", stereo="free")

print("six distinct ligands: native strings", len(native_distinct), "rxembed isomers", len(rx_distinct))
print("six identical ligands: native strings", len(native_identical), "rxembed isomers", len(rx_identical))

## Coordinated ligand stereochemistry

- Native RDKit retains the phosphorus tag below.
- Normal RDKit parsing clears the coordinated amine and carbanion tags.
- rxembed retains all three as ordinary @/@@ stereo in the SMILES core.

In [ ]:
donor_inputs = {
    "P": "[Pd](Cl)(Cl)(Cl)([P@H](C)O)",
    "N": "[Pd](Cl)(Cl)(Cl)([N@H](C)O)",
    "C": "CC[P]1(CC)CC[P](CC)(CC)->[Ni+2]<-12<-[O-]C(=O)N(c1ccccc1)[C@@H-]->2c1ccccc1",
}

native_tags = {}
for symbol, smiles in donor_inputs.items():
    native = Chem.MolFromSmiles(smiles)
    donor = next(
        atom
        for atom in native.GetAtoms()
        if (symbol != "C" and atom.GetSymbol() == symbol) or (symbol == "C" and atom.GetFormalCharge() == -1)
    )
    native_tags[symbol] = donor.GetChiralTag()
    isomer = rx.metal(smiles, "SPL")[0]
    core = rx.cxsmiles(isomer).split(" |", 1)[0]
    print(symbol, "RDKit parsed", donor.GetChiralTag(), "rxembed", isomer.stereo_label, core)

## Fully dative ferrocene does not embed natively

The graph is readable, but RDKit's native bounds construction raises bad lower bound for the ten
dative Fe-C bonds. The next cell shows that failure.

In [ ]:
ferrocene = "[cH]12->[Fe+2]3456789(<-[cH]1[cH]->3[cH-]->4[cH]->52)<-[cH]1[cH]->6[cH]->7[cH-]->8[cH]->91"
native_fc = Chem.AddHs(Chem.MolFromSmiles(ferrocene))
try:
    with rdBase.BlockLogs():
        code = rdDistGeom.EmbedMolecule(native_fc, randomSeed=7)
    print("EmbedMolecule returned:", code, "(no exception raised)")
except RuntimeError as error:
    message = next(line.strip() for line in str(error).splitlines() if "bad lower bound" in line)
    print(f"{type(error).__name__}: {message}")

## Multimetal example: fac to mer without redrawing the complex

The Mn hydride has an active octahedral Mn centre, a ferrocene Fe centre, a configurable amine, and a
carbon stereocentre, all defined by a 3D input. By default rxembed retains the measured Fe, N, and C states
while enumerating the 15 Mn arrangements (6 fac, 9 mer).

- Mn arrangements only: 15.
- Mn plus the N5 pair: 30.
- Mn, N5, C47, and the configurable Fe face: 120.

In [ ]:
mn_states = rx.metal("structures/mnh.xyz", center="Mn")
mn_n_states = rx.metal("structures/mnh.xyz", center="Mn", stereo={"N5": "racemic"})
all_states = rx.metal("structures/mnh.xyz", center="all", stereo="racemic")
mn_states.summary()
print("candidate counts", {"Mn": len(mn_states), "Mn + N5": len(mn_n_states), "all configurable": len(all_states)})
print(f"fac {len(mn_states.filter(label='fac'))} | mer {len(mn_states.filter(label='mer'))}")
print("stereo labels present:", {isomer.stereo_label for isomer in mn_states})

### Select only the chemistry wanted

Only Mn arrangements and N5 vary here. With this many distinct donors, `label` ('mer'), `hand` ('lambda'),
and N5 `stereo` alone still match several arrangements, so `arrangement=` pins the exact one too (the same
fragility `07_metal.ipynb` notes for cis/trans). The carbon centre and ferrocene face stay as measured.

In [ ]:
chosen = mn_n_states.select(
    center="Mn", label="mer", hand="lambda", stereo="N5:R", arrangement="P2 N6 N5 C64 H67 C65"
)
chosen.summary(details=True)
print(chosen.label, chosen.chirality, chosen.stereo_label)

### Store, read, and re-embed the complete state

Plain SMILES can carry the dative graph, but not this complete multimetal arrangement. The
canonical CXSMILES records both metal states, haptic winding, and ligand stereo. Reading it returns
one selected isomer, and embedding the string reproduces that state.

In [ ]:
written = rx.cxsmiles(chosen)
loaded = rx.metal(written, center="all")
print(f"{len(loaded)} isomer(s) loaded | round-trips to the written CXSMILES:", rx.cxsmiles(loaded[0]) == written)

embedded = rx.embed(written, n=1, seed=7).minimize(max_iters=2000)
for report in embedded.check().values():
    print(report.summary())

print(written)

## Status

| capability | RDKit native | rxembed on RDKit |
|---|---|---|
| store SP/TB/OH tags | works | reads for diagnostics |
| enforce SP/TB/OH during DG | unreliable and silently unchecked | checks selected state immediately after DG |
| symmetry-reduced metal isomers | manual permutation handling | enumerated from ligand graph and polyhedron symmetry |
| coordinated C/N/P stereo | partial | retained, selected, and written as @/@@ |
| haptic and ferrocene state | no complete embedding representation | centroid sites plus canonical face winding |
| multimetal arrangement string | plain SMILES lacks the state | canonical dative CXSMILES fixed point |
| final energy | UFF is incomplete for transition metals | restrained UFF cleanup, then optional xTB |

rxembed does not replace RDKit or make UFF a transition-metal energy model. It adds explicit
coordination identity, enumeration, constraints, serialization, and validation around RDKit's graph
and conformer machinery.